# AgStack DPI — End-to-End Grant Lifecycle, Programmatically

This notebook runs the complete DPI field-access story as plain HTTP calls (`requests`), with an assertion after every step — if it executes top to bottom, the demo works. It is the programmatic companion to `demo_e2e.sh` and the script for the recorded backup run.

**The story in one line:** a farmer's field gets a neutral name (GeoID), the world sees only a privacy mask (L0), ownership itself is a credential the farmer self-issues, a second signed, purpose-bound, revocable credential goes to one buyer — and only credentials open exact geometry (L1), verified offline at the node, with the hub routing but never authorizing.

**Standards on display** (for EU dataspace reviewers): SD-JWT VC (OWF/EUDI-compatible) · W3C StatusList2021 revocation · ODRL-style purpose limitation · S2-cell L0 masking · EUDR GeoJSON export (RFC 7946, 6-decimal) · Merkle-root ListIDs.

**Prerequisites — three services running locally** (see `demo_guide_local.md`): Hub :8000 · Node :8001 (Behind Hub :8000) (`JWKS_URL`, `AR_TRUSTED_ISSUER_PUBKEY`) · Pancake :8100 (`PANCAKE_ISSUER_KEY`, `HUB_JWKS_URL`).

Run unattended: `jupyter nbconvert --to notebook --execute dpi_demo_e2e.ipynb`

In [21]:
import base64, json, time, zlib
import requests

HUB     = "http://127.0.0.1:8000"
NODE    = "http://127.0.0.1:8000"
PANCAKE = "http://127.0.0.1:8100"

FARMER = {"email": "farmer@demo.agstack.org", "password": "Demo#Farmer1", "first_name": "Flora", "last_name": "Farmer", "phone": "+50400000001", "role": "farmer", "country": "USA"}
BUYER  = {"email": "buyer@demo.agstack.org",  "password": "Demo#Buyer1",  "first_name": "Bruno", "last_name": "Buyer",  "phone": "+3200000002"}

DEMO_WKT = ("POLYGON((-119.48387145996094 36.40810420514039,-119.48382854461671 36.40449287165274,-119.4750738143921 36.40452743066338,-119.4751811027527 36.408052369005034,-119.48387145996094 36.40810420514039))")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))

def show(title, obj, keys=None):
    print(f"--- {title} ---")
    if keys and isinstance(obj, dict):
        obj = {k: obj.get(k) for k in keys}
    print(json.dumps(obj, indent=2, default=str)[:1200])

PASSED = 0
def check(label, condition):
    global PASSED
    assert condition, f"FAIL: {label}"
    PASSED += 1
    print(f"PASS [{PASSED}]: {label}")

for name, url in [("hub", HUB), ("node", NODE), ("pancake", PANCAKE)]:
    r = requests.get(f"{url}/docs", timeout=5)
    check(f"{name} reachable on {url}", r.status_code == 200)

PASS [1]: hub reachable on http://127.0.0.1:8000
PASS [2]: node reachable on http://127.0.0.1:8000
PASS [3]: pancake reachable on http://127.0.0.1:8100


## Step 1 — Identity: two parties, one trust anchor

The hub is the ecosystem's identity provider (RS256 JWTs, public JWKS). We create **two distinct accounts**: the farmer (field owner, grant issuer) and the buyer (grantee). Everything downstream is a transaction *between* them — consent is only meaningful with two parties.

In [22]:
for persona in (FARMER, BUYER):
    r = requests.post(f"{HUB}/users/register", json=persona)
    # 201 = created, 400 = already exists from a previous run: both fine
    check(f"register {persona['email']}", r.status_code in (201, 400))

def login(email, password):
    r = requests.post(f"{HUB}/users/login", data={"username": email, "password": password})
    check(f"login {email}", r.status_code == 200)
    return r.json()["access_token"]

FARMER_JWT = login(FARMER["email"], FARMER["password"])
BUYER_JWT  = login(BUYER["email"],  BUYER["password"])

header = json.loads(b64url_decode(FARMER_JWT.split(".")[0]))
show("JWT header (note RS256 + kid — verifiable against the hub's public JWKS)", header)
check("hub signs RS256 with kid", header.get("alg") == "RS256" and "kid" in header)

jwks = requests.get(f"{HUB}/.well-known/jwks.json").json()
check("hub JWKS published", len(jwks.get("keys", [])) >= 1)

PASS [4]: register farmer@demo.agstack.org
PASS [5]: register buyer@demo.agstack.org
PASS [6]: login farmer@demo.agstack.org
PASS [7]: login buyer@demo.agstack.org
--- JWT header (note RS256 + kid — verifiable against the hub's public JWKS) ---
{
  "alg": "RS256",
  "kid": "hub-key-1",
  "typ": "JWT"
}
PASS [8]: hub signs RS256 with kid
PASS [9]: hub JWKS published


## Step 2 — Register the field: a neutral name, not a data grab

The AR node converts a boundary into a **GeoID** — a deterministic hash of the S2 cell cover. Same boundary, same ID, computed by anyone; the registry stores geometry and country, **never ownership**. Like DNS: naming is neutral infrastructure. Registering is also how you *claim* a field — the claim itself lives in Step 5, not in the registry.

In [23]:
r = requests.post(f"{NODE}/register-field-boundary",
                  json={"wkt": DEMO_WKT, "threshold": 95, "return_s2_indices": False},
                  headers={"Authorization": f"Bearer {FARMER_JWT}"})
body = r.json()
if r.status_code == 200:
    GEOID = body.get("Geo Id") or body["matched geo ids"][0]
else:  # 400 = already registered from a previous run — extract the existing ID
    detail = body.get("detail", {})
    GEOID = detail.get("Geo Id") or detail.get("matched geo ids", [None])[0]
check("field registered (GeoID obtained)", isinstance(GEOID, str) and len(GEOID) == 64)
print("GeoID:", GEOID)
print("GeoID (short):", body.get("Geo Id Short") or (body.get("detail") or {}).get("Geo Id Short"))

PASS [10]: field registered (GeoID obtained)
GeoID: f34d8f0df673de29bf128dd6e3112f86652126df00cd26ec637feb5042d6b05c
GeoID (short): 595d410e97d96789


## Step 3 — The public view is a mask (L0)

Anyone can resolve a GeoID — and gets an **S2 Level-10 cell** (~7–10 km), the country, and a rounded area. Privacy is a property of the architecture, not a policy promise. Note we send **no credentials at all**.

In [24]:
r = requests.get(f"{NODE}/fetch-field/{GEOID}")   # no headers whatsoever
l0 = r.json()
show("Anonymous view", l0, keys=["MaskingLevel", "Geo Data", "message"])
check("anonymous caller gets L0", r.status_code == 200 and l0["MaskingLevel"] == "L0")
check("no exact geometry in L0 payload", "wkt" not in json.dumps(l0.get("Geo Data", {})).lower())

r = requests.get(f"{NODE}/fetch-field/{GEOID}", headers={"X-Field-Grant": "garbage-not-a-credential"})
check("garbage credential degrades to L0 (never an error, never L1)", r.json()["MaskingLevel"] == "L0")

--- Anonymous view ---
{
  "MaskingLevel": "L0",
  "Geo Data": {
    "cell_token": "8094df",
    "country": "United States",
    "area_ha": 31.0
  },
  "message": "Field fetched successfully (masked)."
}
PASS [11]: anonymous caller gets L0
PASS [12]: no exact geometry in L0 payload
PASS [13]: garbage credential degrades to L0 (never an error, never L1)


## Step 4 — Ownership lives with the farmer, not the registry

The farmer files the GeoID into a **FieldList** on Pancake — a federated, self-hostable service (this data never touches hub or registry). The ListID is the **Merkle root** of the member GeoIDs: deterministic, and membership is provable with an inclusion proof.

In [25]:
r = requests.post(f"{PANCAKE}/fieldlists",
                  json={"name": "Flora's demo farm", "geoids": [GEOID]},
                  headers={"Authorization": f"Bearer {FARMER_JWT}"})
check("fieldlist created (hub token accepted by Pancake — zero glue)", r.status_code == 201)
LIST_ID = r.json()["list_id"]
print("ListID (Merkle root):", LIST_ID)

r2 = requests.post(f"{PANCAKE}/fieldlists",
                   json={"name": "Flora's demo farm", "geoids": [GEOID]},
                   headers={"Authorization": f"Bearer {FARMER_JWT}"})
check("ListID is deterministic (same fields → same root)", r2.json()["list_id"] == LIST_ID)

proof = requests.get(f"{PANCAKE}/fieldlists/{LIST_ID}/proof/{GEOID}",
                     headers={"Authorization": f"Bearer {FARMER_JWT}"}).json()
show("Merkle inclusion proof", proof)

PASS [14]: fieldlist created (hub token accepted by Pancake — zero glue)
ListID (Merkle root): 595d410e97d967894088889097cbe53ff86537205f4bb4539689118708cf09e8
PASS [15]: ListID is deterministic (same fields → same root)
--- Merkle inclusion proof ---
{
  "geoid": "f34d8f0df673de29bf128dd6e3112f86652126df00cd26ec637feb5042d6b05c",
  "list_id": "595d410e97d967894088889097cbe53ff86537205f4bb4539689118708cf09e8",
  "proof": []
}


## Step 5 — The owner is the first grantee

Here is the model's defining move: **even the owner reaches her own geometry through a credential.** A hub login answers *who are you*; only a grant answers *what may you see*. So Flora self-issues an **owner grant** (`purpose: "owner"`, 365 days, auto-renewal configurable) — ownership is not a registry flag, it's the first credential. Transfer, inheritance, and delegation thereby become auditable credential operations.

In [26]:
# A valid hub login WITHOUT a credential sees only the mask:
r = requests.get(f"{NODE}/fetch-field/{GEOID}", headers={"Authorization": f"Bearer {FARMER_JWT}"})
check("login != authorization: farmer's JWT alone gets L0", r.json()["MaskingLevel"] == "L0")

# The owner grant: an ordinary grant, issued to oneself.
r = requests.post(f"{PANCAKE}/grants/issue",
                  json={"list_id": LIST_ID, "grantee_account": FARMER["email"],
                        "purpose": "owner", "validity_days": 365},
                  headers={"Authorization": f"Bearer {FARMER_JWT}"})
check("owner grant self-issued", r.status_code == 201)
OWNER_JTI = r.json()["jti"]

r = requests.get(f"{PANCAKE}/grants/received", headers={"Authorization": f"Bearer {FARMER_JWT}"})
OWNER_CREDENTIAL = next(g["credential"] for g in r.json() if g.get("jti") == OWNER_JTI)

r = requests.get(f"{NODE}/fetch-field/{GEOID}", headers={"X-Field-Grant": OWNER_CREDENTIAL})
check("owner's own credential earns L1 — even the owner uses the rail",
      r.json()["MaskingLevel"] == "L1")

PASS [16]: login != authorization: farmer's JWT alone gets L0
PASS [17]: owner grant self-issued
PASS [18]: owner's own credential earns L1 — even the owner uses the rail


## Step 6 — Consent as a signed, machine-readable contract

Now the farmer shares with someone else: a **grant to the buyer** — an SD-JWT verifiable credential, Ed25519-signed by this Pancake instance, carrying a **purpose limitation** (`eudr-due-diligence`), an expiry, and a pointer into a public revocation list. ODRL-style usage control in a portable artifact — not a checkbox in someone's database.

In [27]:
r = requests.post(f"{PANCAKE}/grants/issue",
                  json={"list_id": LIST_ID, "grantee_account": BUYER["email"],
                        "purpose": "eudr-due-diligence", "validity_days": 3},
                  headers={"Authorization": f"Bearer {FARMER_JWT}"})
check("grant issued by farmer to buyer", r.status_code == 201)
grant = r.json()
JTI, STATUS_IDX = grant["jti"], grant["status_list_index"]
show("Grant record", grant, keys=["jti", "grantee_account", "purpose", "expires_at", "status", "status_list_index"])

PASS [19]: grant issued by farmer to buyer
--- Grant record ---
{
  "jti": "01KY236NAT4QA8VBCTZPQP7MHF",
  "grantee_account": "buyer@demo.agstack.org",
  "purpose": "eudr-due-diligence",
  "expires_at": "2026-07-24T10:23:04Z",
  "status": "active",
  "status_list_index": 1
}


## Step 7 — The buyer retrieves the credential with a plain account login

No wallet app required: the grantee authenticates with their own DPI account and collects the credential (`GET /grants/received`). The credential is **holder-agnostic** — the same artifact could sit in the TraceFoodChain wallet next sprint, or a future EUDI wallet, without any change to issuer or verifier.

In [28]:
r = requests.get(f"{PANCAKE}/grants/received", headers={"Authorization": f"Bearer {BUYER_JWT}"})
check("buyer retrieved credentials with account login", r.status_code == 200 and len(r.json()) >= 1)
CREDENTIAL = next(g["credential"] for g in r.json() if g["jti"] == JTI)

jwt_part, *disclosures = CREDENTIAL.split("~")
claims = json.loads(b64url_decode(jwt_part.split(".")[1]))
show("Credential claims — the consent contract", claims,
     keys=["iss", "sub", "vct", "purpose", "exp", "status", "_sd_alg"])
disclosed = [json.loads(b64url_decode(d)) for d in disclosures if d]
print("Selective disclosures (salt, claim, value):")
for d in disclosed: print("  ", d[1], "=", d[2][:20] + "...")
check("credential discloses our GeoID", any(d[2] == GEOID for d in disclosed))
check("credential is purpose-bound", claims.get("purpose") == "eudr-due-diligence")

PASS [20]: buyer retrieved credentials with account login
--- Credential claims — the consent contract ---
{
  "iss": "did:web:pancake.agstack.org",
  "sub": "595d410e97d967894088889097cbe53ff86537205f4bb4539689118708cf09e8",
  "vct": "agstack.org/credentials/field-access-grant/v1",
  "purpose": "eudr-due-diligence",
  "exp": 1784888584,
  "status": {
    "status_list": {
      "uri": "http://localhost:8100/grants/status-list",
      "idx": 1
    }
  },
  "_sd_alg": "sha-256"
}
Selective disclosures (salt, claim, value):
   fields.0 = f34d8f0df673de29bf12...
PASS [21]: credential discloses our GeoID
PASS [22]: credential is purpose-bound


## Step 8 — Presentation: the credential alone opens L1

The buyer presents the credential to the node in an `X-Field-Grant` header — **with no login whatsoever**. The node verifies the Ed25519 signature against the accredited issuer key, checks expiry, checks every selective disclosure against the `_sd` digests, checks the revocation bit — all **offline** except the public status list.

In [29]:
r = requests.get(f"{NODE}/fetch-field/{GEOID}", headers={"X-Field-Grant": CREDENTIAL})
l1 = r.json()
check("credential alone earns L1 (no Authorization header sent)",
      r.status_code == 200 and l1["MaskingLevel"] == "L1")
check("exact geometry returned", l1.get("Geo JSON") is not None)

r = requests.get(f"{NODE}/fetch-field-wkt/{GEOID}", headers={"X-Field-Grant": CREDENTIAL})
check("WKT endpoint honors the grant", r.json()["WKT"] is not None)

PASS [23]: credential alone earns L1 (no Authorization header sent)
PASS [24]: exact geometry returned
PASS [25]: WKT endpoint honors the grant


## Step 9 — The same story through the hub gateway

In production, callers reach the federation through the **hub gateway**, which routes each request to the right regional node. The crucial property: the hub *routes* but never *authorizes* — it forwards the `X-Field-Grant` header untouched, and the node still does all verification against the issuer key. Reads are public (L0 is the product); writes require a hub account. Data sovereignty is topology: even in the routed path, the authorization decision is made at the edge, next to the data.

In [30]:
r = requests.get(f"{HUB}/fetch-field/{GEOID}")   # anonymous, via hub
check("hub gateway: anonymous read routed to node → L0", r.status_code == 200 and r.json()["MaskingLevel"] == "L0")

r = requests.get(f"{HUB}/fetch-field/{GEOID}", headers={"X-Field-Grant": CREDENTIAL})
check("hub gateway forwards the credential untouched → node verifies → L1",
      r.status_code == 200 and r.json()["MaskingLevel"] == "L1")

r = requests.post(f"{HUB}/register-field-boundary", json={"wkt": DEMO_WKT, "threshold": 95})
check("hub gateway: anonymous WRITE is refused (401)", r.status_code == 401)

PASS [26]: hub gateway: anonymous read routed to node → L0
PASS [27]: hub gateway forwards the credential untouched → node verifies → L1
PASS [28]: hub gateway: anonymous WRITE is refused (401)


## Step 10 — The EUDR artifact

With L1 access, the buyer exports the **EUDR-profile GeoJSON** — WGS84, 6-decimal precision, RFC 7946 ring winding — the artifact an EU operator files for due diligence. This is the deliverable the whole rail exists to authorize.

In [31]:
r = requests.get(f"{NODE}/geoid/{GEOID}/eudr-export", headers={"X-Field-Grant": CREDENTIAL})
check("EUDR export with grant", r.status_code == 200)
show("EUDR GeoJSON (truncated)", r.json().get("Geo JSON", {}))

r = requests.get(f"{NODE}/geoid/{GEOID}/eudr-export")   # anonymous
check("EUDR export refused without authorization (401)", r.status_code == 401)

PASS [29]: EUDR export with grant
--- EUDR GeoJSON (truncated) ---
{
  "type": "Feature",
  "geometry": {
    "type": "MultiPolygon",
    "coordinates": [
      [
        [
          [
            -119.483871,
            36.408104
          ],
          [
            -119.483829,
            36.404493
          ],
          [
            -119.475074,
            36.404527
          ],
          [
            -119.475181,
            36.408052
          ],
          [
            -119.483871,
            36.408104
          ]
        ]
      ]
    ]
  }
}
PASS [30]: EUDR export refused without authorization (401)


## Step 11 — Revocation: the farmer changes their mind

One call flips a bit in the public **StatusList2021** registry. The *same credential, byte for byte,* now yields only L0 — everywhere in the federation, within seconds, with no key rotation and no callback to the buyer. Consent that cannot be withdrawn is not consent. And revocation is **surgical**: it touches this one grant, nobody else's — the owner's credential keeps working.

In [32]:
r = requests.post(f"{PANCAKE}/grants/revoke", json={"jti": JTI},
                  headers={"Authorization": f"Bearer {FARMER_JWT}"})
check("farmer revoked the grant", r.status_code == 200 and r.json()["status"] == "revoked")

sl = requests.get(f"{PANCAKE}/grants/status-list").json()   # public, no auth
bits = zlib.decompress(b64url_decode(sl["encoded"]))
check("revocation bit is set in the public status list",
      bits[STATUS_IDX // 8] & (1 << (STATUS_IDX % 8)))

t0 = time.time()
r = requests.get(f"{NODE}/fetch-field/{GEOID}", headers={"X-Field-Grant": CREDENTIAL})
check(f"same credential now yields L0 ({time.time()-t0:.2f}s after revocation)",
      r.json()["MaskingLevel"] == "L0")
r = requests.get(f"{NODE}/geoid/{GEOID}/eudr-export", headers={"X-Field-Grant": CREDENTIAL})
check("EUDR export refused post-revocation (401)", r.status_code == 401)

# Revocation is surgical — per-grant, not per-field:
r = requests.get(f"{NODE}/fetch-field/{GEOID}", headers={"X-Field-Grant": OWNER_CREDENTIAL})
check("farmer's owner credential still earns L1 after the buyer's revocation",
      r.json()["MaskingLevel"] == "L1")

PASS [31]: farmer revoked the grant
PASS [32]: revocation bit is set in the public status list
PASS [33]: same credential now yields L0 (0.28s after revocation)
PASS [34]: EUDR export refused post-revocation (401)
PASS [35]: farmer's owner credential still earns L1 after the buyer's revocation


## Step 12 — The audit trail

Every issuance and revocation was appended to **MEAL** — Pancake's Ed25519-signed, hash-chained ledger — queryable per GeoID through the OpenScience Auditing API. Any auditor can verify the consent history without asking permission. (EUDR due diligence wants exactly this provenance.)

In [33]:
r = requests.get(f"{PANCAKE}/audit/{GEOID}", headers={"Authorization": f"Bearer {FARMER_JWT}"})
check("audit trail available for the GeoID", r.status_code == 200)
events = r.json()
show("Audit events (issuance + revocation)", events)

PASS [36]: audit trail available for the GeoID
--- Audit events (issuance + revocation) ---
{
  "geoid": "f34d8f0df673de29bf128dd6e3112f86652126df00cd26ec637feb5042d6b05c",
  "event_count": 6,
  "events": [
    {
      "packet_id": "01KY236MEW1G86DP1FB8RCT7QV",
      "meal_id": "01KY236MESA91K45PC2Y7EMNZQ",
      "sequence_number": 1,
      "time_index": "2026-07-21T10:23:03.641006",
      "author": {
        "agent_id": "farmer@demo.agstack.org",
        "agent_type": "human"
      },
      "event": {
        "event_type": "fieldlist.created",
        "list_id": "595d410e97d967894088889097cbe53ff86537205f4bb4539689118708cf09e8",
        "name": "Flora's demo farm",
        "geoid_count": 1
      },
      "packet_hash": "7579e6168473f6185e541663bb6ccdcab9bc6d676d5ed303fc01082a9acdc5ac",
      "signature": "4bacf48f36216cba544ae8c590884217b9bd44c0fa2cfd878ad60110f33785f46c46c115a28a0cb2b9a5a6866363bbe7cb1b095522d86fc630c1e84f77413e09"
    },
    {
      "packet_id": "01KY236N1A80X5RMM8F

## What just happened

| Step | Standard / principle |
|---|---|
| GeoID | Deterministic naming — neutral registry, no ownership inside |
| L0 mask | Privacy by construction (S2 L10 cell) |
| FieldList | Merkle-root ListID with inclusion proofs |
| Owner grant | **Ownership = the first credential** — registration is the claim; login never authorizes geometry |
| Buyer grant | SD-JWT VC, Ed25519, ODRL-style purpose, expiry |
| Delivery | Account-anchored (wallet optional, EUDI-compatible format) |
| Presentation | Offline verification at the node — the hub routes, never authorizes |
| Hub gateway | Public reads, gated writes, credential pass-through — sovereignty is topology |
| EUDR export | RFC 7946 / 6-decimal / ring-wound GeoJSON |
| Revocation | W3C StatusList2021, public, per-grant (owner unaffected), seconds to propagate |
| Audit | Signed hash-chained ledger, open verification |

Every component seen here — capture, registry, issuer, holder — is a **replaceable reference implementation of an open interface**; only the GeoID namespace, the hub trust registry, and the credential/ListID specs are fixed. That thin waist is the architecture (`doc/dpi_updated_architecture_20260708.md`).

In [34]:
print("=" * 60)
print(f"  ALL {PASSED} CHECKS PASSED — demo verified end to end")
print("=" * 60)

  ALL 36 CHECKS PASSED — demo verified end to end


### Step 2A: Identity Resolution (Deduplication)
Registering an exact duplicate or near duplicate should resolve to the same Geo ID without throwing a 400 error.

### Step 2B: Identity Resolution (Nested Sub-plot)
Registering a small plot fully contained within the large field should register correctly and create a `child_of` mapping in the DB.

In [35]:
print("\n--- Step 2A: Identity Resolution (Duplicate) ---\n")
wkt_dupe = "POLYGON((-119.48387145996094 36.40810420514039,-119.48382854461671 36.40449287165274,-119.4750738143921 36.40452743066338,-119.4751811027527 36.408052369005034,-119.48387145996094 36.40810420514039))"
res_dupe = requests.post(
    f"{HUB}/register-field-boundary",
    headers={"Authorization": f"Bearer {FARMER_JWT}", "Content-Type": "application/json"},
    json={"wkt": wkt_dupe, "threshold": 95, "return_s2_indices": False}
)
data_dupe = res_dupe.json()
print(f"Status: {res_dupe.status_code}")
print(f"Message: {data_dupe.get('message')}")
dupe_geoid = data_dupe.get("Geo Id")
print(f"Original GeoID: {GEOID}, Duplicate GeoID: {dupe_geoid}")
assert dupe_geoid == GEOID, "Deduplication failed!"

print("\n--- Step 2B: Identity Resolution (Nested Sub-plot) ---\n")
wkt_sub = "POLYGON((-119.4800 36.4060, -119.4800 36.4070, -119.4780 36.4070, -119.4780 36.4060, -119.4800 36.4060))"
res_sub = requests.post(
    f"{HUB}/register-field-boundary",
    headers={"Authorization": f"Bearer {FARMER_JWT}", "Content-Type": "application/json"},
    json={"wkt": wkt_sub, "threshold": 95, "return_s2_indices": False}
)
data_sub = res_sub.json()
print(f"Status: {res_sub.status_code}")
print(f"Message: {data_sub.get('message')}")
sub_geoid = data_sub.get("Geo Id")
print(f"Sub-plot GeoID: {sub_geoid}")
assert sub_geoid != GEOID, "Sub-plot got identical GeoID!"



--- Step 2A: Identity Resolution (Duplicate) ---

Status: 200
Message: Exact geometry already registered.
Original GeoID: f34d8f0df673de29bf128dd6e3112f86652126df00cd26ec637feb5042d6b05c, Duplicate GeoID: f34d8f0df673de29bf128dd6e3112f86652126df00cd26ec637feb5042d6b05c

--- Step 2B: Identity Resolution (Nested Sub-plot) ---

Status: 200
Message: Field Boundary registered successfully.
Sub-plot GeoID: 42cd359251d89018149c85af556591b5e8db3b1b91b875f807eef7341be47574
